In [2]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [3]:
950_000/32

29687.5

In [4]:
from collections import defaultdict
from copy import deepcopy
from uuid import uuid4

from src.utils import disk
from src.utils import debug

In [30]:
directory = '_data/001_staged/boxscores/000_normalized/data'
files = disk.listdir(directory)

team_keys = (
    'team_id',
    'team_city',
    'team_name',
    'team_tricode',
    'team_slug'
)

player_keys = (
    'person_id',
    'first_name',
    'family_name',
    'name_i',
    'player_slug',
    # 'position',
    # 'jersey_num',
)

teams_collected = set()
players_collected = set()

for file in files:
    data = disk.read_jsonl(file)
    for boxscore in data:
        teams_collected.add(tuple(boxscore.get(x) for x in team_keys))
        players_collected.add(tuple(boxscore.get(x) for x in player_keys))

In [31]:
import pandas as pd

df_teams = pd.DataFrame(teams_collected, columns=team_keys)
df_players = pd.DataFrame(players_collected, columns=player_keys)

In [32]:
from _v2.utils.provenance import build_provenance_envelope

path_teams  ='_data/001_staged/teams/000_normalized/from_boxscores.json'
path_players  ='_data/001_staged/players/000_normalized/from_boxscores.json'

kwargs = dict(
    path_input_data=directory,
    path_processing_script='_v2/normalize/001_boxscores.ipynb',
    is_directory_input=True,
    is_directory_output=False,
)


disk.write_json(path_teams, build_provenance_envelope(**kwargs, data=df_teams.to_dict(orient='records')))
disk.write_json(path_players, build_provenance_envelope(**kwargs, data=df_players.to_dict(orient='records')))